In [ ]:
# for logistic regression
# Accuracy: 82.12290502793296%
# Precision: 80.88235294117648%
# Recall: 74.32432432432432%

# for GaussianNB
# Accuracy: 80.44692737430168%
# Precision: 71.91011235955057%
# Recall: 86.48648648648648%
# confusion matrix: [[80 25]
#  [10 64]]

# for KNN
# Accuracy: 82.68156424581005%
# Precision: 81.15942028985508%
# Recall: 75.67567567567568%

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [184]:
df = pd.read_csv("titanic_train.csv")
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Age"] = df["Age"].fillna(df.groupby(["Pclass", "Sex"])["Age"].transform("median"))
df["HasCabin"] = df["Cabin"].apply(lambda x: 0 if pd.isna(x) else 1)
df.drop("Cabin", axis=1, inplace=True)
df["Sex"] = df["Sex"].map({"male": 0, "female":1})
df["Embarked"] = df["Embarked"].map({"S":0, "C":1, "Q":2})
df["Title"] = df["Name"].str.extract("([A-Za-z]+)\.", expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
 	'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].map({'Master':1, 'Miss':2, 'Mr':3, 'Mrs':4, 'Rare':5})
# df['AgeBin'] = pd.cut(df['Age'], 5, labels=[0,1,2,3,4])
# df['FareBin'] = pd.qcut(df['Fare'], 4, labels=[0,1,2,3])
# df['AgeBin'] = df['AgeBin'].astype(int)
# df['FareBin'] = df['FareBin'].astype(int)
# df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
# df['IsAlone'] = 0
# df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
# this increased complexity in logistic , good recall and accuracy in GaussianNB,
df.info()
df["Embarked"].nunique()
df.sample(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    int64  
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    int64  
 11  HasCabin     891 non-null    int64  
 12  Title        891 non-null    int64  
dtypes: float64(2), int64(9), object(2)
memory usage: 90.6+ KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,HasCabin,Title
428,429,0,3,"Flynn, Mr. James",0,25.0,0,0,364851,7.7500,2,0,3
537,538,1,1,"LeRoy, Miss. Bertha",1,30.0,0,0,PC 17761,106.4250,1,0,2
446,447,1,2,"Mellinger, Miss. Madeleine Violet",1,13.0,0,1,250644,19.5000,0,0,2
855,856,1,3,"Aks, Mrs. Sam (Leah Rosen)",1,18.0,0,1,392091,9.3500,0,0,4
858,859,1,3,"Baclini, Mrs. Solomon (Latifa Qurban)",1,24.0,0,3,2666,19.2583,1,0,4
726,727,1,2,"Renouf, Mrs. Peter Henry (Lillian Jefferys)",1,30.0,3,0,31027,21.0000,0,0,4
514,515,0,3,"Coleff, Mr. Satio",0,24.0,0,0,349209,7.4958,0,0,3
163,164,0,3,"Calic, Mr. Jovo",0,17.0,0,0,315093,8.6625,0,0,3
849,850,1,1,"Goldenberg, Mrs. Samuel L (Edwiga Grabowska)",1,35.0,1,0,17453,89.1042,1,1,4
396,397,0,3,"Olsson, Miss. Elina",1,31.0,0,0,350407,7.8542,0,0,2


In [203]:
X = df.drop(columns=["PassengerId", "Survived", "Name", "Ticket"])
y = df["Survived"]
# X.sample(10)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
param_grid = {'knn__n_neighbors': [3,5,7,9,11,13]}


classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
)

classifierCV.fit(X_train, y_train)
y_pred = classifierCV.predict(X_test)



In [204]:
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100}%")
print(f"Precision: {precision_score(y_test, y_pred)*100}%")
print(f"Recall: {recall_score(y_test, y_pred)*100}%")
# res = pd.DataFrame(classifierCV.cv_results_)
# print(res)
print(classifierCV.best_params_)
# print(f"confusion matrix: {confusion_matrix(y_test, y_pred)}")

Accuracy: 82.68156424581005%
Precision: 81.15942028985508%
Recall: 75.67567567567568%
{'knn__n_neighbors': 13}
